In [18]:
import boto3

print("📊 INVENTARIO AWS\n")

# =====================
# EC2
# =====================
ec2 = boto3.client('ec2')

print("🖥️ EC2:")

ec2_count = 0

for r in ec2.describe_instances()['Reservations']:
    for i in r['Instances']:
        ec2_count += 1
        print(f"- {i['InstanceId']} | {i['State']['Name']} | {i['InstanceType']}")

if ec2_count == 0:
    print("No hay instancias")

print(f"Total EC2: {ec2_count}")

print("\n" + "="*40 + "\n")

# =====================
# S3
# =====================
s3 = boto3.client('s3')

print("📦 S3:")

buckets = s3.list_buckets()['Buckets']

if not buckets:
    print("No hay buckets")
else:
    for b in buckets:
        print(f"- {b['Name']}")

print(f"Total buckets: {len(buckets)}")

📊 INVENTARIO AWS

🖥️ EC2:
- i-0bd52d6196ffdcb38 | running | t3.micro
Total EC2: 1


📦 S3:
No hay buckets
Total buckets: 0


In [19]:
import boto3

ec2 = boto3.client('ec2')
s3 = boto3.client('s3')
cw = boto3.client('cloudwatch')

ec2_count = sum(len(r['Instances']) for r in ec2.describe_instances()['Reservations'])

print(f"Instancias EC2: {ec2_count}")

# Métrica
cw.put_metric_data(
    Namespace='InventarioAWS',
    MetricData=[{
        'MetricName': 'TotalEC2',
        'Value': ec2_count,
        'Unit': 'Count'
    }]
)

# Alerta simple
if ec2_count == 0:
    print("🚨 ALERTA: No hay instancias")

Instancias EC2: 1


In [20]:
!cat inventario.txt

📊 INVENTARIO AWS

🖥️ EC2:
i-0bd52d6196ffdcb38 | running | t3.micro
Total EC2: 1

📦 S3:
Total buckets: 0


In [23]:
import boto3
from datetime import datetime

ec2 = boto3.client('ec2', region_name='us-west-2')
cloudwatch = boto3.client('cloudwatch', region_name='us-west-2')

# contar instancias running
running = 0

for r in ec2.describe_instances()['Reservations']:
    for i in r['Instances']:
        if i['State']['Name'] == 'running':
            running += 1

# enviar métrica
cloudwatch.put_metric_data(
    Namespace='MiApp',
    MetricData=[
        {
            'MetricName': 'EC2Running',
            'Value': running,
            'Unit': 'Count'
        }
    ]
)

print(f"✅ EC2 corriendo: {running}")

✅ EC2 corriendo: 1


In [24]:
import boto3
import json
from datetime import datetime
from botocore.exceptions import ClientError

print("🔍 INVENTARIO AWS COMPLETO + ALERTAS CLOUDWATCH")
print("=" * 70)
print(f"🕒 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ===========================================
# CONFIGURACIÓN
# ===========================================
REGION = 'us-west-2'
inventario = {}

# Clientes AWS
ec2 = boto3.client('ec2', region_name=REGION)
s3 = boto3.client('s3', region_name=REGION)
cloudwatch = boto3.client('cloudwatch', region_name=REGION)

# ===========================================
# 1. INVENTARIO EC2
# ===========================================
print("\n☁️  EC2 INSTANCIAS")
print("-" * 40)

try:
    ec2_resp = ec2.describe_instances()
    ec2_count = 0
    
    for res in ec2_resp['Reservations']:
        for inst in res['Instances']:
            print(f"ID: {inst['InstanceId']:<20} Estado: {inst['State']['Name']:<10} Tipo: {inst['InstanceType']}")
            ec2_count += 1
    
    inventario['ec2'] = ec2_count
    print(f"📊 Total EC2: {ec2_count}")
    
except Exception as e:
    print(f"❌ Error EC2: {e}")

# ===========================================
# 2. INVENTARIO S3
# ===========================================
print("\n🗄️  BUCKETS S3")
print("-" * 30)

try:
    s3_resp = s3.list_buckets()
    s3_count = len(s3_resp['Buckets'])
    
    for bucket in s3_resp['Buckets']:
        print(f"✅ {bucket['Name']}")
    
    inventario['s3_buckets'] = s3_count
    print(f"📊 Total Buckets: {s3_count}")
    
except Exception as e:
    print(f"❌ Error S3: {e}")

# ===========================================
# 3. MÉTRICAS CLOUDWATCH PERSONALIZADAS
# ===========================================
print("\n📈 CREANDO MÉTRICAS PERSONALIZADAS")
print("-" * 45)

try:
    # Métrica 1: Total Recursos AWS
    total_recursos = sum(inventario.values())
    cloudwatch.put_metric_data(
        Namespace='InventarioAWS/Custom',
        MetricData=[{
            'MetricName': 'TotalRecursos',
            'Value': total_recursos,
            'Unit': 'Count',
            'Timestamp': datetime.utcnow()
        }]
    )
    print(f"✅ Métrica 'TotalRecursos': {total_recursos}")

    # Métrica 2: EC2 Running
    running_ec2 = sum(1 for res in ec2_resp['Reservations'] 
                     for inst in res['Instances'] 
                     if inst['State']['Name'] == 'running')
    
    cloudwatch.put_metric_data(
        Namespace='InventarioAWS/Custom',
        MetricData=[{
            'MetricName': 'EC2Running',
            'Value': running_ec2,
            'Unit': 'Count'
        }]
    )
    print(f"✅ Métrica 'EC2Running': {running_ec2}")
    
except Exception as e:
    print(f"❌ Error CloudWatch: {e}")

# ===========================================
# 4. RESUMEN INVENTARIO
# ===========================================
print("\n📋 RESUMEN INVENTARIO")
print("-" * 30)
print(json.dumps(inventario, indent=2))
print(f"\n💾 Total Recursos: {sum(inventario.values())}")

# ===========================================
# 5. CONFIGURAR ALARMA (OPCIONAL)
# ===========================================
print("\n🚨 CONFIGURANDO ALARMA CLOUDWATCH")
print("-" * 40)

try:
    cloudwatch.put_metric_alarm(
        AlarmName='Alerta-EC2-Many',
        MetricName='EC2Running',
        Namespace='InventarioAWS/Custom',
        Statistic='Average',
        Period=300,
        EvaluationPeriods=1,
        Threshold=5.0,
        ComparisonOperator='GreaterThanThreshold',
        AlarmDescription='Alerta si hay más de 5 EC2 running'
    )
    print("✅ Alarma 'Alerta-EC2-Many' creada")
    
except Exception as e:
    print(f"⚠️  Alarma ya existe o error: {e}")

print("\n" + "="*70)
print("✅ INVENTARIO COMPLETADO + MÉTRICAS PUBLICADAS")
print(f"🕒 Fin: {datetime.now().strftime('%H:%M:%S')}")


🔍 INVENTARIO AWS COMPLETO + ALERTAS CLOUDWATCH
🕒 2026-03-19 00:02:30

☁️  EC2 INSTANCIAS
----------------------------------------
ID: i-0bd52d6196ffdcb38  Estado: running    Tipo: t3.micro
📊 Total EC2: 1

🗄️  BUCKETS S3
------------------------------
📊 Total Buckets: 0

📈 CREANDO MÉTRICAS PERSONALIZADAS
---------------------------------------------


/tmp/ipykernel_21993/889333082.py:76: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'Timestamp': datetime.utcnow()


✅ Métrica 'TotalRecursos': 1
✅ Métrica 'EC2Running': 1

📋 RESUMEN INVENTARIO
------------------------------
{
  "ec2": 1,
  "s3_buckets": 0
}

💾 Total Recursos: 1

🚨 CONFIGURANDO ALARMA CLOUDWATCH
----------------------------------------
✅ Alarma 'Alerta-EC2-Many' creada

✅ INVENTARIO COMPLETADO + MÉTRICAS PUBLICADAS
🕒 Fin: 00:02:31
